In [7]:
# -----------------------
#  CELL 1 CONFIG SECTION
# -----------------------

CONFIG = {
    "output_mode": "combined",  # or "separate"
    "output_filename": "../postproc_outputs/final_table.tex",  # used if output_mode is "combined"
    "output_dir": "../postproc_outputs/generated_tables",      # used if output_mode is "separate"

    # Paths for LaTeX components
    "preamble_path": "../postproc_inputs/latex_preamble.tex",
    "fragment_template_path": "../postproc_inputs/latex_superpanel_fragment.tex.j2",
    "closing_path": "../postproc_inputs/latex_closing.tex",

    "superpanels": [
        {"model": "llama", "prompt": "zeroShot", "standard": "gold"},
        {"model": "gpt", "prompt": "zeroShot", "standard": "gold"},
        {"model": "llama", "prompt": "zeroShotCoT", "standard": "gold"},
        {"model": "gpt", "prompt": "zeroShotCoT", "standard": "gold"},
        {"model": "llama", "prompt": "zeroShot", "standard": "silver"},
        {"model": "gpt", "prompt": "zeroShot", "standard": "silver"},
        {"model": "llama", "prompt": "zeroShotCoT", "standard": "silver"},
        {"model": "gpt", "prompt": "zeroShotCoT", "standard": "silver"},
    ]
}


In [8]:
# -----------------------
#  CELL 2 - IMPORTS and LOAD TEMPLATE
# -----------------------

import os
import re
from pathlib import Path

import pandas as pd
from jinja2 import Environment, FileSystemLoader

# Extract template path from config
fragment_template_path = Path(CONFIG["fragment_template_path"])
template_dir = str(fragment_template_path.parent)
template_name = fragment_template_path.name

# LaTeX escape filter (for labels like subgroup names)
LATEX_MAP = {
    '\\': r'\textbackslash{}', '&': r'\&', '%': r'\%',
    '$': r'\$', '#': r'\#', '_': r'\_', '{': r'\{', '}': r'\}',
    '~': r'\textasciitilde{}', '^': r'\textasciicircum{}'
}
_LATEX_RE = re.compile('|'.join(re.escape(k) for k in LATEX_MAP))
def latex_escape(s):
    return _LATEX_RE.sub(lambda m: LATEX_MAP[m.group()], '' if s is None else str(s))

# Set up Jinja2 environment with whitespace control
env = Environment(
    loader=FileSystemLoader(template_dir),
    trim_blocks=True,
    lstrip_blocks=True
)
env.filters['tex'] = latex_escape

fragment_template = env.get_template(template_name)
print(f"Loaded LaTeX fragment template: {template_name} from {template_dir}")


Loaded LaTeX fragment template: latex_superpanel_fragment.tex.j2 from ..\postproc_inputs


In [9]:
# -----------------------
# CELL 3: Metric Extraction Function (Safe for LaTeX)
# -----------------------

def extract_metrics(model, prompt, standard):
    """
    Load all relevant pretty CSVs and structure them for one superpanel.
    Matches real file layout: files per metric and subgroup, prompt methods as columns.
    Returns: dict[(extract, subgrp_type, subgrp_val)] → metric_name → value (str)
    """
    metric_names = ["qwk", "pearson", "icc3", "mae", "rmse", "deltaT", "w"]
    base_dir = Path(f"../postproc_intermediate/matrices/{model}/{standard}/aggregate")

    metrics_by_subgroup = {}

    for metric in metric_names:
        matches = list(base_dir.glob(f"{metric}_*_*_pretty.csv"))

        for fpath in matches:
            parts = fpath.stem.split("_")
            if len(parts) < 4:
                continue
            _, subgrp_type, subgrp_val, _ = parts  # e.g., deltaT_inst_MGB_pretty

            df = pd.read_csv(fpath, index_col=[0, 1])
            df.columns.name = None

            if prompt not in df.columns:
                continue

            df = df.reset_index()
            for _, row in df.iterrows():
                extract = row["extract"]
                key = (extract, subgrp_type, subgrp_val)
                if key not in metrics_by_subgroup:
                    metrics_by_subgroup[key] = {}
                val = row[prompt]
                if pd.isnull(val):
                    val = "--"
                metrics_by_subgroup[key][metric] = str(val)

    return metrics_by_subgroup


In [10]:
# -----------------------
# CELL 4: Render Superpanel to LaTeX
# -----------------------

def render_superpanel(model, prompt, standard, metrics_by_subgroup, is_first):
    """
    Renders a LaTeX superpanel using the provided metrics and Jinja2 template.

    Special rule:
    - For GPT + Full Note: render ONLY the adversarial row ("adv"),
      suppress overall/inst/time subgroups.
    """
    # Pretty labels
    model_label = "LLaMA" if str(model).lower() == "llama" else "GPT"
    prompt_map = {
        "zeroShot": "Zero-Shot",
        "zeroShotCoT": "Zero-Shot Chain-of-Thought",
        "fewShot1": "Few-Shot (k=1)",
        "fewShot2": "Few-Shot (k=2)",
        "fewShot3": "Few-Shot (k=3)",
        "fewShot4": "Few-Shot (k=4)",
    }
    prompt_label = prompt_map.get(prompt, str(prompt))

    # Desired order and pretty extract labels
    extract_order = ["manual", "automatic", "fullNote"]
    extract_pretty = {"manual": "Manual", "automatic": "Automatic", "fullNote": "Full Note"}
    subgroup_order = ["inst", "time", "adv"]

    sections = []
    for extract in extract_order:
        section = {
            "extract": extract,
            "extract_label": extract_pretty[extract],
            "overall": {},
            "subgroups": []
        }

        # Special case: GPT + Full Note → only show adversarial row
        if model.lower() == "gpt" and extract == "fullNote":
            adv_key = (extract, "adv", "abbrReintro")
            if adv_key in metrics_by_subgroup:
                section["subgroups"].append({
                    "type": "adv",
                    "value": "abbrReintro",
                    "metrics": metrics_by_subgroup[adv_key]
                })
            # Do not set section["overall"]
            sections.append(section)
            continue

        # Normal case: overall row
        overall_key = (extract, "overall", "all")
        section["overall"] = metrics_by_subgroup.get(overall_key, {})

        # Normal case: subgroups
        for subgrp_type in subgroup_order:
            subgroup_keys = [k for k in metrics_by_subgroup.keys()
                             if k[0] == extract and k[1] == subgrp_type]
            for k in sorted(subgroup_keys, key=lambda x: str(x[2]).lower()):
                section["subgroups"].append({
                    "type": k[1],
                    "value": k[2],
                    "metrics": metrics_by_subgroup[k]
                })

        sections.append(section)

    # Render with pretty labels for banner
    rendered = fragment_template.render(
        model=model,
        prompt=prompt,
        standard=standard,
        is_first=is_first,
        sections=sections,
        model_label=model_label,
        prompt_label=prompt_label,
    )
    return rendered


In [11]:
# -----------------------
# CELL 5: Write Final Output(s)
# -----------------------

def write_outputs(superpanels_latex):
    """
    Write LaTeX superpanels to file(s) based on CONFIG.
    Includes preamble and closing files if output_mode == 'combined'.
    
    Args:
        superpanels_latex: list of (name, latex_string)
    """
    output_mode = CONFIG["output_mode"]

    # Load preamble and closing once
    preamble = Path(CONFIG["preamble_path"]).read_text(encoding="utf-8")
    closing = Path(CONFIG["closing_path"]).read_text(encoding="utf-8")

    if output_mode == "combined":
        output_path = Path(CONFIG["output_filename"])
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(preamble)
            f.write("\n\n")
            for idx, (_, latex) in enumerate(superpanels_latex):
                f.write(latex)
                f.write("\n")
                if idx < len(superpanels_latex) - 1:
                    f.write("\\pagebreak\n\n")
            f.write(closing)
        print(f"✅ Combined LaTeX written to: {output_path}")

    elif output_mode == "separate":
        output_dir = Path(CONFIG["output_dir"])
        output_dir.mkdir(parents=True, exist_ok=True)

        for name, latex in superpanels_latex:
            filename = f"{name}.tex"
            path = output_dir / filename
            with open(path, "w", encoding="utf-8") as f:
                f.write(preamble)
                f.write("\n\n")
                f.write(latex)
                f.write("\n")
                f.write(closing)
            print(f"✅ Saved: {path}")

    else:
        raise ValueError(f"Unknown output_mode: {output_mode}")


In [12]:
# -----------------------
# CELL 6: Master Driver Loop
# -----------------------

superpanels_latex = []

# Track whether we've emitted the super-column header for each standard
emitted_header = {"gold": False, "silver": False}

for panel_cfg in CONFIG["superpanels"]:
    model = panel_cfg["model"]
    prompt = panel_cfg["prompt"]
    standard = panel_cfg["standard"]

    print(f"🔹 Considering: {model} / {prompt} / {standard}")

    # Load and extract metrics
    metrics_by_subgroup = extract_metrics(model, prompt, standard)

    # Skip panels with no data at all (prevents placeholder GPT panes)
    if not metrics_by_subgroup:
        print("   ↪ Skipping (no data found for this panel).")
        continue

    # Emit the super-column header only for the first *rendered* panel per standard
    is_first = not emitted_header.get(standard, False)

    # Render LaTeX
    latex_code = render_superpanel(
        model=model,
        prompt=prompt,
        standard=standard,
        metrics_by_subgroup=metrics_by_subgroup,
        is_first=is_first
    )

    # Mark header as emitted for this standard
    emitted_header[standard] = True

    # Save for writing
    name = f"{model}_{prompt}_{standard}"
    superpanels_latex.append((name, latex_code))

# Write to disk based on CONFIG
write_outputs(superpanels_latex)


🔹 Considering: llama / zeroShot / gold
🔹 Considering: gpt / zeroShot / gold
🔹 Considering: llama / zeroShotCoT / gold
🔹 Considering: gpt / zeroShotCoT / gold
🔹 Considering: llama / zeroShot / silver
🔹 Considering: gpt / zeroShot / silver
🔹 Considering: llama / zeroShotCoT / silver
🔹 Considering: gpt / zeroShotCoT / silver
✅ Combined LaTeX written to: ..\postproc_outputs\final_table.tex
